**Contexte**

Une entreprise dispose de données provenant de plusieurs bâtiments équipés de capteurs. Pour chaque observation, on dispose par exemple de la température moyenne, de l’humidité, du nombre d'occupants, de l’heure, du jour de la semaine, de la surface du bâtiment et de la consommation énergétique. L'objectif de l’atelier est de construire avec TensorFlow/Keras un réseau de neurones capable de prédire la consommation énergétique d'un bâtiment à partir de caractéristiques telles que température, humidité et nombre d'occupants.

**Partie 0 – mise en place de l’environnement**

Alors la structure du projet est déja fais la création du notebook donc nous allons a la troisiéme question qui est l'installation des bibliothéques et leurs importations

3) Installer et importer tensorflow, matplotlib et numpy

In [2]:
!pip install tensorflow
!pip install matplotlib
!pip install numpy

In [4]:
# import des bibliothéques

import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

print("importation terminée")


importation terminée


**Partie 1 – Génération du dataset**

1) Générer aléatoirement 1000 valeurs pour chacune des variables suivantes :

a) temperature : valeurs qui suivent une loi normale avec une moyenne de 25 °C et un
écart-type de 4 °C.

In [22]:
# alors ici on utilise la loi normale avec une moyenne de 25 (loc) et un écart-type de 4 (scale) et une size de 1000
#donc on vas utiliser la syntaxe random.normale()

n=1000
temperature = tf.random.normal(shape=(1000,) , mean=25 , stddev=4 )
temperature.shape
print(tf.round(temperature[:10] * 100)/100)



tf.Tensor([27.3  25.73 20.88 20.58 25.02 25.26 21.31 17.92 25.9  20.31], shape=(10,), dtype=float32)


b) humidite : valeurs réparties de façon uniforme entre 30 % et 80 %.

In [23]:
humidite = tf.random.uniform(shape=(1000,) , minval=30 , maxval=80)
print(tf.round(humidite[:10] * 100)/100)

tf.Tensor([66.13 70.94 36.12 74.78 53.88 55.31 69.09 32.18 31.4  71.33], shape=(10,), dtype=float32)


c) occupants : valeurs entières choisies entre 1 et 49 inclus.

In [25]:
# c'est une valeur entier choisie entre 1 et 49 on utilise on ajoute le type que est un entier ici

occupants = tf.random.uniform(shape=(1000,), minval=1, maxval=50, dtype=tf.int32)

print(occupants[:10].numpy())


[25 41 40 19 34 33 29 17  2 44]


2) Déterminer la variable consommation avec la formule suivante :

a) la consommation de base (0 °C, pas d'humidité et pièce vide) est de 50

La formule donnée par l'énoncé est **linéaire** (chaque variable contribue indépendamment et de
façon proportionnelle à la consommation) :

$$
\text{consommation} = 50 \;+\; 5 \times \text{temperature} \;+\; 1{,}5 \times \text{humidite}
\;+\; 4 \times \text{occupants} \;+\; \varepsilon
$$

où :
- **50** : la consommation de base incompressible (0 °C, 0 % d'humidité, pièce vide) ;
- **5 × température** : chaque degré supplémentaire coûte 5 unités (chauffage/climatisation) ;
- **1,5 × humidité** : chaque point d'humidité en plus coûte 1,5 unité (déshumidification) ;
- **4 × occupants** : chaque personne présente coûte 4 unités (éclairage, appareils, etc.) ;
- **ε (bruit)** : un terme aléatoire `~ N(0, 10)` (généré ici aussi avec `tf.random.normal`) qui simule tous les facteurs non mesurés
  (isolation du bâtiment, appareils allumés par hasard, courants d'air...). Sans ce bruit, la
  relation serait **parfaitement linéaire** et un simple réseau à un seul neurone (voire une
  régression linéaire) suffirait à obtenir une erreur nulle : le bruit rend le problème réaliste et
  justifie l'usage d'un modèle capable d'apprendre malgré l'incertitude.

In [26]:
n=1000
bruit = tf.random.normal(shape=(n,), mean=0, stddev=10)

# occupants est en int32 : on le convertit en float32 avant de l'utiliser dans un calcul
# avec des tenseurs float32 (temperature, humidite, bruit), sinon TensorFlow lève une erreur
# de type (contrairement à NumPy, TensorFlow ne convertit jamais les types implicitement).
occupants_float = tf.cast(occupants, tf.float32)

consommation = (
    50
    + 5 * temperature
    + 1.5 * humidite
    + 4 * occupants_float
    + bruit
)

print("dtype de consommation :", consommation.dtype)
print("\nAperçu des 10 premières valeurs de consommation :")
print(tf.round(consommation[:10] * 100) / 100)
print(f"\nConsommation moyenne : {tf.reduce_mean(consommation):.2f}")
print(f"Consommation min/max : {tf.reduce_min(consommation):.2f} / {tf.reduce_max(consommation):.2f}")


dtype de consommation : <dtype: 'float32'>

Aperçu des 10 premières valeurs de consommation :
tf.Tensor([396.29 438.44 369.09 334.1  398.91 388.53 378.35 259.97 234.36 447.43], shape=(10,), dtype=float32)

Consommation moyenne : 359.72
Consommation min/max : 193.37 / 512.88


3) Rassembler les variables (temperature, humidite et occupants) dans la matrice des
caractéristiques (features) X de taille 1000x3 en convertissant éventuellement les données au
format (float32) optimisé pour les calculs

Un réseau de neurones Keras attend en entrée une **matrice 2D** de forme
`(nombre d'observations, nombre de caractéristiques)`, ici `(1000, 3)`. Avec des tenseurs
TensorFlow, on empile `temperature`, `humidite` et `occupants_float` **en colonnes** avec
`tf.stack(..., axis=1)` (l'équivalent, côté TensorFlow, de `np.column_stack`) : chaque tenseur 1D
de forme `(1000,)` devient une colonne du tenseur 2D résultant.

On s'assure que le résultat est bien en **`float32`** (type déjà utilisé par tous les tenseurs
générés, sauf `occupants` qu'on convertit d'abord) car c'est :
- le type de données **par défaut utilisé en interne par TensorFlow/Keras** pour les poids et les
  calculs ;
- deux fois plus économe en mémoire que le `float64` ;
- **optimisé pour le calcul sur GPU**, généralement bien plus rapide en 32 bits.

In [27]:
X = tf.stack([temperature, humidite, occupants_float], axis=1)

print("Forme de X :", X.shape)
print("dtype de X :", X.dtype)
print("\nAperçu des 5 premières lignes de X (colonnes : temperature, humidite, occupants) :")
print(X[:5].numpy()) # comme on l'as vue dans le veille .numpy permet d'avoir le résultat format tableau numpy


Forme de X : (1000, 3)
dtype de X : <dtype: 'float32'>

Aperçu des 5 premières lignes de X (colonnes : temperature, humidite, occupants) :
[[27.30032  66.13454  25.      ]
 [25.733604 70.93983  41.      ]
 [20.876245 36.118126 40.      ]
 [20.578606 74.780426 19.      ]
 [25.018553 53.876625 34.      ]]
